# Intermediate 02 - Dispatcher와 라우팅

이 노트북에서는:
- Dispatcher 모듈의 동작 원리를 이해합니다
- 다중 노드 환경에서의 라우팅 로직을 실습합니다
- `%%kamcmd` 매직으로 실제 서버 상태를 조회합니다

---

## Dispatcher란?

```
         ┌── Node 1 (10.0.0.1) ── 📱
📞 ──── 🖥️ LB ──┤
         └── Node 2 (10.0.0.2) ── 📱
```

- LB(Load Balancer)가 들어오는 SIP 요청을 여러 Node에 분배
- `ds_select_dst()`가 대상 노드를 선택
- 알고리즘: round-robin(0), hash(2), first(4) 등

## 1. Dispatcher 함수 호출 시뮬레이션

실제 Kamailio에서 `ds_select_dst("1", "4")`를 호출하면:
1. Set 1의 destination 목록에서
2. 알고리즘 4(first available)로 대상을 선택
3. `$du` (destination URI)를 설정

In [ ]:
%%sip INVITE
From: <sip:1001@example.com>;tag=disp1
To: <sip:1002@example.com>

In [ ]:
# Dispatcher로 대상 선택
ds_select_dst("1", "4");
xlog("Selected destination via dispatcher");

## 2. 실제 서버 상태 조회 (%%kamcmd)

로컬 Kamailio가 실행 중이면 `%%kamcmd`로 실시간 상태를 볼 수 있습니다.

> **주의**: 로컬 Kamailio가 실행 중이어야 합니다. 실행 중이 아니면 에러가 납니다.

In [ ]:
# Dispatcher 상태 조회
%%kamcmd dispatcher.list

In [ ]:
# 등록된 사용자 위치 조회
%%kamcmd ul.dump

## 3. 전체 라우팅 플로우 시뮬레이션

초기 INVITE의 전형적인 라우팅 흐름을 시뮬레이션합니다:

```
INVITE 수신 → 메서드 확인 → 다이얼로그 내 여부 확인 →
Dispatcher 선택 → Record-Route → Relay
```

In [ ]:
%%sip INVITE
From: <sip:1001@10.60.91.199>;tag=flow1
To: <sip:1002@10.60.80.218>
Contact: <sip:1001@192.168.1.100:5060>

In [ ]:
# Step 1: 메서드 확인
if (!is_method("INVITE")) {
    send_reply(405, "Method Not Allowed");
    exit;
}
xlog("Step 1: INVITE confirmed");

In [ ]:
# Step 2: 다이얼로그 내 요청인지 확인
if (has_totag()) {
    xlog("In-dialog request — relay directly");
    route(RELAY);
} else {
    xlog("Initial INVITE — need routing");
}

In [ ]:
# Step 3: 대상 사용자 위치 조회
xlog("Looking up $(ru{uri.user})");
lookup("location");

# Step 4: Record-Route (이후 BYE 등도 LB를 거치도록)
record_route();

# Step 5: 릴레이
xlog("Relaying to $ru");
t_relay();

## 4. REGISTER 처리 플로우

REGISTER는 사용자의 현재 위치(IP:port)를 서버에 등록하는 메시지입니다.

In [ ]:
%%sip REGISTER
From: <sip:1001@example.com>;tag=reg1
To: <sip:1001@example.com>
Contact: <sip:1001@192.168.1.100:5060;expires=3600>

In [ ]:
# REGISTER 처리
if (is_method("REGISTER")) {
    xlog("REGISTER from $(fu{uri.user}) at $ct");
    save("location");
    send_reply(200, "OK");
}

## 5. Kamailio 인스턴스 제어 (v0.3)

로컬 개발 환경에서 Kamailio를 시작/중지할 수 있습니다.

In [ ]:
# 현재 상태 확인
%%kamailio status

In [ ]:
# 전체 시작
%%kamailio start

In [ ]:
# 변수 변경 사항 추적
%%diff

---

### 요약

- `ds_select_dst()`로 Dispatcher에서 대상 노드 선택
- `%%kamcmd`로 실제 Kamailio 상태 조회
- `%%kamailio status|start|stop`로 인스턴스 제어
- `%%diff`로 변수 변경 추적

다음 노트북: **Advanced/01-dialog-and-failover.ipynb** →